# 04 — Complete Pipeline: Baselines → Train → Evaluate → Animate

This notebook runs the **entire RL Battery Thermal Management pipeline** end-to-end in one Colab session.

| Section | What it runs |
|---|---|
| **2 — Baselines** | All 5 classical controllers on all 4 heat profiles |
| **3 — PI detail view** | Full time-series plot from `visualize_pack_controller_cooling_3d.py` |
| **4 — PPO training** | On-policy PPO with VecNormalize (100 000-step demo) |
| **5 — SAC training** | Off-policy SAC with replay buffer (100 000-step demo) |
| **6 — All-controller comparison** | PPO, SAC, and all 5 classical controllers head-to-head |
| **7 — 2D animation** | Matplotlib layer-heatmap + signal-trace GIF |
| **8 — 3D PyVista animation** | Cylinder-pack render coloured by temperature |

> **Training note:** 100 000 steps verifies the pipeline but does not produce a well-trained agent.
> For fully-trained models (3 000 000 steps) run `01_train_ppo_colab.ipynb` and
> `02_train_sac_colab.ipynb`, then update the model paths in Section 6.

All outputs are written to `outputs/` inside the cloned repo and displayed inline.  
No Google Drive is required.

## 1 — Setup

**What this section does:**  
Clones (or pulls) the project repository from GitHub, installs every Python dependency listed in
`requirements.txt` (NumPy, Gymnasium, Stable-Baselines3, PyVista, imageio, etc.), confirms the GPU
runtime, runs a project-structure diagnostic, and verifies that the thermal environment can be
imported without errors.

Run all cells here once at the start of every session before anything else.

In [ ]:
# Replace YOUR_USERNAME with your GitHub username before running.
REPO_URL = "https://github.com/Takyi-Wontumi/RL-Battery-Thermal-Management.git"
REPO_DIR = "/content/RL-Battery-Thermal-Management"

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
%cd /content/RL-Battery-Thermal-Management
!pip install -q -r requirements.txt
print("Dependencies installed.")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU.  Runtime → Change runtime type → T4 GPU.")

In [ ]:
!pwd
!find envs -maxdepth 2 -type f | sort
!find configs -maxdepth 2 -type f | sort
!grep -R "from envs" -n training
!grep -R "from configs" -n training

In [ ]:
!PYTHONPATH=. python -c "
from envs.battery_pack_thermal_env_3d import BatteryPackThermalEnv3D
from configs.pack_config import CellConfig, PackConfig
env = BatteryPackThermalEnv3D()
obs, info = env.reset()
env.close()
print('Environment: OK')
print('Observation shape:', obs.shape)
print('Action space:     ', env.action_space)
print('Observation space:', env.observation_space)
"

## 2 — Baseline Controllers

**What this section does:**  
`scripts/compare_pack_baselines_3d.py` runs five deterministic (non-learning) controllers on all
four heat load profiles and saves summary statistics, temperature traces, and per-profile layer
heatmaps to `outputs/`.

**Controllers compared:**

| Controller | Strategy |
|---|---|
| No cooling | `u = 0` always — worst-case thermal runaway reference |
| Constant u = 0.5 | Fixed half-power cooling regardless of temperature |
| Constant u = 1.0 | Fixed full-power cooling — upper bound on cooling energy |
| Bang-bang | `u = 1` when `T_max > threshold`, `u = 0` otherwise |
| Proportional | `u = Kp × (T_max − T_target)`, clipped to [0, 1] |

**Heat profiles:**

| Profile | Characteristic |
|---|---|
| UniformConstant | All 24 cells generate identical constant heat |
| NonuniformStep | Corner cells generate 1.5× more heat than interior cells |
| PulsedHotspot | One random cell pulses to 3× heat at random intervals |
| RandomNonuniform | Per-cell heat drawn from a uniform distribution each episode |

**Why this matters:**  
These five controllers establish the performance floor and ceiling for RL agents.  A well-trained
PPO or SAC agent should match PI on temperature safety while using less cooling energy than
constant u = 1.0, because it has learned *when* cooling is actually needed.

In [ ]:
!PYTHONPATH=. python -m scripts.compare_pack_baselines_3d

In [ ]:
import glob
import pandas as pd
from IPython.display import Image as IPyImage, display

csv_path = "outputs/phase2_3d_baseline_summary.csv"
if os.path.exists(csv_path):
    df_bl = pd.read_csv(csv_path)
    print("Baseline summary (mean across all profiles):")
    cols = [c for c in ["total_reward", "T_max_peak_C", "T_gradient_max_C",
                         "time_above_safe_s", "total_cooling_effort"] if c in df_bl.columns]
    display(df_bl.groupby("controller")[cols].mean()
              .sort_values("total_reward", ascending=False).round(2))

for p in sorted(glob.glob("outputs/phase2_3d_baseline_*.png")):
    print(p)
    display(IPyImage(p, width=900))

for p in sorted(glob.glob("outputs/phase2_3d_layer_heatmaps_*.png")):
    print(p)
    display(IPyImage(p, width=900))

## 3 — PI Controller Detail View

**What this section does:**  
`scripts/visualize_pack_controller_cooling_3d.py` runs a single full episode with the PI controller
and produces a four-panel figure:

1. **Top row:** T_max, T_avg, and cooling command `u(t)` over the full 1800-second episode
2. **Bottom row:** Temperature gradient `T_max − T_min` and spatial snapshots of the 3D temperature
   field at regular intervals

**Why PI is the classical benchmark:**  
The Proportional-Integral controller eliminates the steady-state tracking error that pure proportional
control suffers from.  Its gains (Kp, Ki) were tuned by `scripts/tune_pack_pi_controller.py` to
minimise a cost combining temperature overshoot and cooling energy.  It serves as the primary
comparison target for the RL agents.

> To visualise a different controller, edit `controller_key` in the `main()` function of
> `scripts/visualize_pack_controller_cooling_3d.py`.  Valid keys: `no_cooling`, `constant_05`,
> `constant_10`, `bang_bang`, `proportional`, `pi_tuned`, `ppo_best`, `sac_best`.

In [ ]:
# Runs the PI controller (controller_key = 'pi_tuned' is the default in main())
!PYTHONPATH=. python -m scripts.visualize_pack_controller_cooling_3d

In [ ]:
for p in sorted(glob.glob("outputs/3d_visualize_*.png")):
    print(p)
    display(IPyImage(p, width=900))

## 4 — PPO Training

**What this section does:**  
`training/train_pack_ppo_3d.py` trains a **Proximal Policy Optimization** agent on the 3D battery
pack thermal environment.

**How PPO works in this project:**

- **On-policy:** The agent interacts with the environment to collect rollouts, then updates its policy
  once on those rollouts before discarding them.  Fresh data every update prevents stale gradients.
- **VecNormalize:** Temperature observations are normalised online to zero mean and unit variance.
  This makes learning stable regardless of the actual temperature scale or unit system.
- **Observation:** 7-element vector `[T_max, T_avg, T_min, ΔT, Var, T_center, u_prev]` — all
  temperatures normalised by `(safe_temp − target_temp)`.  `T_center` is the geometric centre cell,
  which is hardest to cool and the earliest hotspot indicator.
- **Reward:** Safety term `(T_max − T_safe)²` (quadratic, unnormalised) dominates.  The agent is
  also penalised for unnecessary cooling effort (`0.02 u²`) and action chattering (`0.01 Δu²`).
- **Checkpoints:** Saved every 50 000 steps so training can resume after a Colab disconnect.

> **This demo uses 100 000 steps** — enough to verify the pipeline and produce a partially-trained
> agent.  For a deployable agent use `01_train_ppo_colab.ipynb` with 3 000 000 steps (~1–2 h on T4).

In [ ]:
PPO_SAVE_DIR = "/content/models/ppo_3d_pack"
PPO_LOG_DIR  = "/content/logs/ppo_3d_pack"
os.makedirs(PPO_SAVE_DIR, exist_ok=True)
os.makedirs(PPO_LOG_DIR,  exist_ok=True)
print("PPO models →", PPO_SAVE_DIR)
print("PPO logs   →", PPO_LOG_DIR)

In [ ]:
# 100 000 steps, 4 parallel envs — ~5 min on T4 GPU, ~25 min on CPU.
# Reduce --timesteps 10000 for a 1-minute smoke test.
!PYTHONPATH=. python training/train_pack_ppo_3d.py \
  --timesteps 100000 \
  --n-envs 4 \
  --save-dir {PPO_SAVE_DIR} \
  --log-dir  {PPO_LOG_DIR} \
  --device auto

In [ ]:
print("PPO output files:")
for root, dirs, files in os.walk(PPO_SAVE_DIR):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), PPO_SAVE_DIR)
        size_kb = os.path.getsize(os.path.join(root, f)) // 1024
        print(f"  {rel}  ({size_kb} KB)")

## 5 — SAC Training

**What this section does:**  
`training/train_pack_sac_3d.py` trains a **Soft Actor-Critic** agent on the same environment.

**How SAC differs from PPO:**

| Property | PPO | SAC |
|---|---|---|
| On / off-policy | On-policy | Off-policy |
| Data reuse | Single rollout, discard | Replay buffer (500 000 steps) |
| Exploration | Clipped probability ratio | Entropy maximisation |
| Observation normalisation | VecNormalize (required) | Raw observations |
| Sample efficiency | More steps needed | More sample-efficient |

SAC's entropy bonus (`α × H(π)`) encourages the agent to maintain action diversity throughout
training, which prevents it from collapsing to a deterministic strategy too early.  This is
particularly useful in thermal control, where the optimal response to a temperature spike depends
on context the agent may not have seen yet.

The `--learning-starts 10000` flag means SAC collects 10 000 random steps before the first gradient
update, ensuring the replay buffer contains a diverse range of transitions.

> **This demo uses 100 000 steps.**  For a deployable agent use `02_train_sac_colab.ipynb`
> with 3 000 000 steps.

In [ ]:
SAC_SAVE_DIR = "/content/models/sac_3d_pack"
SAC_LOG_DIR  = "/content/logs/sac_3d_pack"
os.makedirs(SAC_SAVE_DIR, exist_ok=True)
os.makedirs(SAC_LOG_DIR,  exist_ok=True)
print("SAC models →", SAC_SAVE_DIR)
print("SAC logs   →", SAC_LOG_DIR)

In [ ]:
# 100 000 steps — ~5 min on T4 GPU.
!PYTHONPATH=. python training/train_pack_sac_3d.py \
  --timesteps 100000 \
  --save-dir {SAC_SAVE_DIR} \
  --log-dir  {SAC_LOG_DIR} \
  --device auto

In [ ]:
print("SAC output files:")
for root, dirs, files in os.walk(SAC_SAVE_DIR):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), SAC_SAVE_DIR)
        size_kb = os.path.getsize(os.path.join(root, f)) // 1024
        print(f"  {rel}  ({size_kb} KB)")

## 6 — All-Controller Comparison

**What this section does:**  
`evaluation/compare_controllers.py` runs every controller — all 5 classical controllers plus PPO and
SAC — under **identical** conditions (same 4 heat profiles, same random seed sequence) and records
per-episode metrics to a CSV.  Summary bar charts and line plots are saved alongside the CSV.

**Metrics recorded:**

| Metric | Meaning |
|---|---|
| `total_reward` | Cumulative episode reward (higher is better) |
| `T_max_peak_C` | Highest temperature reached by any cell (lower is safer) |
| `T_gradient_max_C` | Maximum temperature difference across the pack |
| `time_above_safe_s` | Seconds any cell spent above the 45 °C safe limit |
| `total_cooling_effort` | ∫ u(t) dt — total energy spent on cooling |

PPO and SAC are loaded from the model files trained in sections 4 and 5.  If a model file is missing
the script skips that controller gracefully — classical controllers always run.

> **With 100 000 training steps** the RL agents will likely underperform PI on `total_reward`.
> After 3 000 000 steps they should reduce `time_above_safe_s` to near zero and use less cooling
> energy than constant u = 1.0.

In [ ]:
PPO_MODEL   = f"{PPO_SAVE_DIR}/ppo_pack_final.zip"
PPO_VECNORM = f"{PPO_SAVE_DIR}/vec_normalize.pkl"
SAC_MODEL   = f"{SAC_SAVE_DIR}/sac_pack_final.zip"
RESULTS_DIR = "outputs/comparison"

os.makedirs(RESULTS_DIR, exist_ok=True)

for label, path in [("PPO model",   PPO_MODEL),
                    ("PPO vecnorm", PPO_VECNORM),
                    ("SAC model",   SAC_MODEL)]:
    status = "found  " if os.path.exists(path) else "MISSING"
    print(f"[{status}] {label}: {path}")

In [ ]:
# --episodes 8  =  2 rounds x 4 profiles.  Increase for tighter error bars.
!PYTHONPATH=. python evaluation/compare_controllers.py \
  --ppo-model   {PPO_MODEL} \
  --ppo-vecnorm {PPO_VECNORM} \
  --sac-model   {SAC_MODEL} \
  --results-dir {RESULTS_DIR} \
  --episodes 8 \
  --seed 7

In [ ]:
csv_path = f"{RESULTS_DIR}/comparison_metrics.csv"
if os.path.exists(csv_path):
    df_cmp = pd.read_csv(csv_path)
    cols = [c for c in ["total_reward", "T_max_peak_C", "T_gradient_max_C",
                         "time_above_safe_s", "total_cooling_effort"] if c in df_cmp.columns]
    summary = (df_cmp.groupby("controller")[cols]
                     .mean()
                     .sort_values("total_reward", ascending=False)
                     .round(2))
    print("Mean metrics across all profiles and rounds:\n")
    display(summary)

for p in sorted(glob.glob(f"{RESULTS_DIR}/*.png")):
    print(p)
    display(IPyImage(p, width=900))

## 7 — 2D Animated Temperature Field

**What this section does:**  
`scripts/animate_pack_temperature_field_3d.py` runs a full 1800-second episode for each requested
controller and records the 3D temperature field at every `--stride`-th simulation step.  It then
renders an animated GIF with two panels side by side:

**Left panel — layer heatmaps:**  
One 2D heatmap per z-layer of the 4 × 3 × 2 pack (rows = x, columns = y, colour = temperature).
The colour scale is fixed to `[ambient_temp, safe_temp]` — blue is cool, red is hot — so all frames
and all controllers share the same scale and can be compared directly.

**Right panel — signal traces:**  
Static T_max, T_avg, and T_gradient traces for the entire episode, with a moving red vertical
time-marker showing the current frame.  A third sub-panel shows the cooling command `u(t)`.

**Stride and FPS:**  
`--stride 15` captures one frame every 15 simulation seconds.  For a 1800-second episode this gives
120 frames.  `--fps 8` means the animation plays over 15 seconds.  Lower stride → smoother
animation, larger file size, longer render time.

In [ ]:
# PI always runs. PPO and SAC are skipped gracefully if their models are not found.
!PYTHONPATH=. python -m scripts.animate_pack_temperature_field_3d \
  --PI --PPO --SAC \
  --profile NonuniformStep \
  --stride 15 \
  --fps 8

In [ ]:
# Show final-frame PNGs first (fast to load), then animated GIFs.
for p in sorted(glob.glob("outputs/3d_animate_*_final_frame.png")):
    print("Final frame:", p)
    display(IPyImage(p, width=900))

for g in sorted(glob.glob("outputs/3d_animate_*.gif")):
    print("Animation:", g)
    display(IPyImage(g, width=900))

## 8 — 3D PyVista Pack Render

**What this section does:**  
`scripts/animate_pack_3d_pyvista.py` produces a GIF of the actual 3D battery pack geometry using
PyVista, showing each physical cell as a coloured cylinder.

**Architecture — why this is different from section 7:**  
Section 7 shows temperature as a flat 2D heatmap per layer — useful for reading exact values but
abstract.  This section renders the *physical geometry*: a 4 × 3 × 2 grid of 18 mm-diameter,
65 mm-long cylindrical cells positioned on the same grid as the thermal model.  You can see the
pack in 3D and immediately identify which physical cells are overheating.

**Implementation details:**
- Each `pv.Cylinder` uses the geometry from `CellConfig` (diameter = 18 mm, length = 65 mm),
  spaced 2 mm apart (`cell_spacing_m` from `PackConfig`).
- Cylinders are oriented along the z-axis (`direction = (0, 0, 1)`) — the same axis as cell length
  in the thermal model.
- Temperature scalars are mapped to `RdYlBu_r`: blue = ambient (25 °C), red = safe limit (45 °C).
- All 24 meshes are built once.  Per-frame updates only change the scalar array in-place
  (`mesh[scalar] = new_values`) then call `plotter.write_frame()` — no geometry is reconstructed.
- `off_screen=True` allows rendering in Colab without a physical display.

**Headless rendering:**  
The next cell installs `xvfb` and calls `pyvista.start_xvfb()`.  This provides a virtual framebuffer
that some PyVista backends need even in off-screen mode.

In [ ]:
!apt-get install -qq xvfb

import pyvista as pv
try:
    pv.start_xvfb()
    print("xvfb started — PyVista headless rendering enabled.")
except Exception as e:
    print(f"xvfb start skipped ({e}).  off_screen=True will still work.")

In [ ]:
# PI always runs.  PPO and SAC are skipped gracefully if models are not found.
!PYTHONPATH=. python -m scripts.animate_pack_3d_pyvista \
  --PI --PPO --SAC \
  --profile NonuniformStep \
  --stride 15 \
  --fps 8

In [ ]:
pyvista_gifs = sorted(glob.glob("outputs/3d_pyvista_*.gif"))
print(f"Displaying {len(pyvista_gifs)} PyVista 3D GIFs:")
for g in pyvista_gifs:
    print(" ", g)
    display(IPyImage(g, width=900))

## Summary

This notebook has run the complete pipeline from raw physics to trained agents:

| Section | Script called | Output |
|---|---|---|
| 2 — Baselines | `scripts/compare_pack_baselines_3d.py` | `outputs/phase2_3d_baseline_*.png`, summary CSV |
| 3 — PI detail | `scripts/visualize_pack_controller_cooling_3d.py` | `outputs/3d_visualize_*.png` |
| 4 — PPO | `training/train_pack_ppo_3d.py` | `/content/models/ppo_3d_pack/` |
| 5 — SAC | `training/train_pack_sac_3d.py` | `/content/models/sac_3d_pack/` |
| 6 — Comparison | `evaluation/compare_controllers.py` | `outputs/comparison/` |
| 7 — 2D animation | `scripts/animate_pack_temperature_field_3d.py` | `outputs/3d_animate_*.gif` |
| 8 — 3D PyVista | `scripts/animate_pack_3d_pyvista.py` | `outputs/3d_pyvista_*.gif` |

**Next steps for better-trained agents:**
- Run `01_train_ppo_colab.ipynb` with `--timesteps 3000000` and save models to Google Drive
- Run `02_train_sac_colab.ipynb` with `--timesteps 3000000`
- Come back to Section 6 and update `PPO_MODEL` / `SAC_MODEL` to the Drive paths
- Try `--profile PulsedHotspot` or `RandomNonuniform` in sections 7–8 for harder heat scenarios